# Data Preprocessing
Preparing the data for modeling, based on findings in `01_exploration.ipynb`, including dropping specific problematic columns and imputing missing values.

### Import Libraries and Load Data

In [3]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [4]:
# read csv
df = pd.read_csv('data/initial/Combined_Housing_Data.csv')

# filter for Residential and SingleFamilyResidence
data = df[(df['PropertyType'] == 'Residential') & (df['PropertySubType'] == 'SingleFamilyResidence')]

# drop Property type columns
data = data.drop(columns=['PropertyType', 'PropertySubType'])

In [5]:
# create CloseMonth 
data['CloseDate'] = pd.to_datetime(data['CloseDate'])
data['CloseMonth'] = data['CloseDate'].dt.to_period('M')
data['CloseYear'] = data['CloseDate'].dt.year

# Distribution of records per year
print(f"Date range: {data['CloseDate'].min().date()} → {data['CloseDate'].max().date()}")
print(f"Unique months: {data['CloseMonth'].nunique()}")
print(f"\nRows per year:")
print(data['CloseYear'].value_counts().sort_index().to_string())

Date range: 2025-12-01 → 2026-05-31
Unique months: 6

Rows per year:
CloseYear
2025     20910
2026    102544


### Dropping Duplicates and Problematic Columns

From `01_exploration.ipynb`, we gathered various observations about the columns and their contents:
* Multiple columns, including but not limited to `CoveredSpaces`, `MiddleOrJuniorSchoolDistrict`, and `TaxYear`, were completely empty and will be dropped
* There are multiple columns which are sparse, such as `MiddleOrJuniorSchool` and `SubdivisionName`, which will also be dropped 
* There are various administrative columns, such as `BuyerAgentFirstName` and `ListingKeyNumeric`, that do not provide any predictive value
* `ListPrice`, `OriginalListPrice`, and `DaysOnMarket` are all columns that are either highly correlated with `ClosePrice` or can only be known after sale, all of which induce data leakage
* `StateOrProvince` contains some listings from Arizona, which must be removed before modeling
* Although almost all boolean "YN" columns (i.e. `WaterfrontYN`, `AttachedGarageYN`) contain numerous missing values, they do not need to be dropped as their missing values can be imputed later
* About half of the dataset contains duplicate listings (have the same `ListingKey`), which means half the data will need to be dropped

In [6]:
# drop duplicate values in dataset, based on ListingKey
before = len(data)
data = data.drop_duplicates(subset='ListingKey', keep='first')
after = len(data)
print(f'Rows before any alterations: {before:,} rows')
print(f'Dropped {before - after:,} rows ({(before - after) / before * 100:.2f}%)')
print(f'Shape: {data.shape}')

Rows before any alterations: 123,454 rows
Dropped 61,760 rows (50.03%)
Shape: (61694, 78)


In [7]:
# filter out any AZ records and keep only CA records
data = data[data['StateOrProvince'] == 'CA']

# admininistrative cols
admin_keywords = ['Agent', 'Office', 'Listing', 'Address', 'Email', 
                  'AOR', 'Mls', 'MLS', 'Buyer', 'Flooring', 'Street', 'Contract', 'State', 'List', 'Business', 'Builder', 'Frequency']
admin_cols = [col for col in data.columns if any(keyword in col for keyword in admin_keywords)]

# columns that are 100% empty
empty_cols = ['MiddleOrJuniorSchoolDistrict', 'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'TaxYear', 'ElementarySchoolDistrict', 'CoveredSpaces', 'BusinessType']

# add columns that are majority empty (avoid YN columns; will be imputed later)
empty_cols += ['BelowGradeFinishedArea', 'BuildingAreaTotal', 'MiddleOrJuniorSchool', 'ElementarySchool', 'HighSchool', 'SubdivisionName', 'Flooring', 'LotSizeDimensions']

# data leakage columns (highly correlated with target or impossible to know until after sale)
data_leak = ['ListPrice', 'OriginalListPrice', 'DaysOnMarket']

# combine all droppped columns into one list
drop_cols = set(list(admin_cols + empty_cols + data_leak))
data = data.drop(columns=[col for col in drop_cols if col in data.columns])

In [8]:
data.columns

Index(['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'CloseDate',
       'ClosePrice', 'Latitude', 'Longitude', 'LivingArea', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'Levels', 'LotSizeArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode',
       'AssociationFee', 'LotSizeSquareFeet', 'CloseMonth', 'CloseYear'],
      dtype='object')

### Outlier Capping
From `01_exploration.ipynb`, there are multiple features with outliers that significantly skew the data and/or make no sense whatsoever. These outliers will need to be accounted for before any models can use this data. Furthermore `ClosePrice` will be log-transformed into `logClosePrice` to normalize the target, allowing linear models to make more accurate predictions.

**Note:** `LotSizeSquareFeet` will be capped later on through a percentile, but this will be done after the train/test splits have been established to avoid data leakage.

In [9]:
# bounds for specific features
bounds = {
    'ClosePrice': (10_000, 3_840_000),
    'BathroomsTotalInteger': (0, 6),
    'GarageSpaces': (0, 10),
    'LivingArea': (0, 15_000),
    # 'LotSizeSquareFeet': (0, train_data['LotSizeSquareFeet'].quantile(0.995)), # cap on training data quantile
    'BedroomsTotal': (0, 10),
    'ParkingTotal': (0, 10)
}

# perform outlier capping
print(f'Rows before outlier capping: {len(data)}')
for col, (low, high) in bounds.items():
    data = data[(data[col] >= low) & (data[col] <= high)]
    print(f'Filtered values in {col} between {low:,} and {high:,}')
print(f'Rows after outlier capping: {len(data):,} rows')

# additionally, log transform ClosePrice into logClosePrice (more appropriate for Linear Regression)
data['logClosePrice'] = np.log1p(data['ClosePrice'])

# drop ClosePrice from the model, use logClosePrice instead 
data = data.drop(columns=['ClosePrice'])

Rows before outlier capping: 61693
Filtered values in ClosePrice between 10,000 and 3,840,000
Filtered values in BathroomsTotalInteger between 0 and 6
Filtered values in GarageSpaces between 0 and 10
Filtered values in LivingArea between 0 and 15,000
Filtered values in BedroomsTotal between 0 and 10
Filtered values in ParkingTotal between 0 and 10
Rows after outlier capping: 56,824 rows


### Handling Missing Values
Now that the problematic columns have been removed, the remaining missing values must be dealt with.

In [10]:
# visualize missing values for data after dropping columns
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Show columns that have at least some missing data
missing_nonzero = missing_df[missing_df['Missing Count'] > 0]
print(f"Columns with missing values: {len(missing_nonzero)} out of {len(data.columns)} columns")
missing_nonzero.head(30)

Columns with missing values: 20 out of 30 columns


,Missing Count,Missing %
WaterfrontYN,56796,100.0
BasementYN,55630,97.9
MainLevelBedrooms,20409,35.9
AssociationFee,14544,25.6
HighSchoolDistrict,13676,24.1
AttachedGarageYN,5680,10.0
Stories,5532,9.7
ViewYN,5229,9.2
NewConstructionYN,4439,7.8
Levels,4204,7.4


Even after dropping problematic columns, there are still numerous columns with empty values that need to be accounted for. 

**Here are some strategies for imputing the missing data:**

| Feature | Strategy | Reasoning |
|---|---|---|
| YN columns | Impute with False | Assume missing values mean feature is absent |
| `Latittude` and `Longitude` | Impute with county specific medians | Imputing based on specific geographic information |
| `AssociationFee` and `GarageSpaces` | Impute with 0 | Assume missing values mean feature is absent |
| `MainLevelBedrooms` | Impute with median and add indicator variable | With heavily right skewed features, median is robust and indicator variables tell the model whether the value was imputed or already present |
| `Stories` | Impute with median or mode (both are 1) | Low cardinality discrete feature |
| `LotSizeSquareFeet` and `LotSizeAcres` | Cross-calculate based on counterpart value, impute remaining missing values with median | Since both of these columns are linearly related, one can be calculated from the other | 
| `City` | Impute with "unknown" | Want to preserve it as a category |

Multiple features still need to be cleaned before imputing can happen, such as `ParkingTotal` (with a minimum value of -16, which does not make sense)

In [11]:
# cross-compute missing values in LotSizeSquareFeet based on counterpart value
# (will only use LotSizeSquareFeet for model, as both columns are perfectly correlated)
sqft_missing = data['LotSizeSquareFeet'].isna() & data['LotSizeAcres'].notna()
data.loc[sqft_missing, 'LotSizeSquareFeet'] = data.loc[sqft_missing, 'LotSizeAcres'] * 43560
print(f'# of cross-computed rows in LotSizeSquareFeet: {sqft_missing.sum():,} rows')

# drop LotSizeAcres and LotSizeArea, as it is highly correlated with LotSizeSquareFeet 
data = data.drop(columns=['LotSizeAcres', 'LotSizeArea'])

# of cross-computed rows in LotSizeSquareFeet: 2 rows


In [12]:
data.columns

Index(['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'CloseDate',
       'Latitude', 'Longitude', 'LivingArea', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'CloseMonth', 'CloseYear', 'logClosePrice'],
      dtype='object')

In [13]:
# impute missing values as per plan above
fill_false = [col for col in data.columns if col.endswith('YN')]
fill_median = ['MainLevelBedrooms', 'Stories', 'LivingArea', 'BathroomsTotalInteger', 'YearBuilt', 
               'LotSizeSquareFeet', 'GarageSpaces']
fill_zero =['AssociationFee', 'ParkingTotal']
fill_unknown = ['City', 'CountyOrParish', 'HighSchoolDistrict']

# fillna in boolean columns
for col in fill_false:
    n_empty = data[col].isna().sum()
    data[col] = data[col].fillna(False)
    if n_empty > 0: 
        print(f'Filled {n_empty:,} rows of {col} with False')

# fillna in median columns
for col in fill_median:
    n_empty = data[col].isna().sum()
    data[col] = data[col].fillna(data[col].median())
    if n_empty > 0: 
        print(f'Filled {n_empty:,} rows of {col} with the median')

# fillna in zero columns
for col in fill_zero:
    n_empty = data[col].isna().sum()
    data[col] = data[col].fillna(0)
    if n_empty > 0: 
        print(f'Filled {n_empty:,} rows of {col} with 0')

# fillna in unknown categorical columns
for col in fill_unknown:
    n_empty = data[col].isna().sum()
    data[col] = data[col].fillna('Unknown')
    if n_empty > 0: 
        print(f'Filled {n_empty:,} rows of {col} with "Unknown"')

# fillna in Levels with "One" (mode value) for now (will be target encoded later)
data['Levels'] = data['Levels'].fillna('One')

# standardize PostalCode to 5 digit format, and fill any missing values with '00000' (will be target encoded later)
if 'PostalCode' in data.columns:
    n = data['PostalCode'].isna().sum()
    data['PostalCode'] = data['PostalCode'].astype(str).str.split('-').str[0].str.strip()
    data.loc[data['PostalCode'].isin(['nan', '', 'None']), 'PostalCode'] = '00000'
    print(f"Filled PostalCode NaN with '00000' and normalized to 5-digit ZIP: {n:,} rows")

Filled 5,229 rows of ViewYN with False
Filled 56,796 rows of WaterfrontYN with False
Filled 55,630 rows of BasementYN with False
Filled 3,776 rows of PoolPrivateYN with False
Filled 5,680 rows of AttachedGarageYN with False
Filled 12 rows of FireplaceYN with False
Filled 4,439 rows of NewConstructionYN with False
Filled 20,409 rows of MainLevelBedrooms with the median
Filled 5,532 rows of Stories with the median
Filled 8 rows of YearBuilt with the median
Filled 1,016 rows of LotSizeSquareFeet with the median
Filled 14,544 rows of AssociationFee with 0
Filled 14 rows of City with "Unknown"
Filled 13,676 rows of HighSchoolDistrict with "Unknown"
Filled PostalCode NaN with '00000' and normalized to 5-digit ZIP: 1 rows


In [14]:
# visualize missing values for data after dropping columns
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# visualize any remaining missing values
missing_nonzero = missing_df[missing_df['Missing Count'] > 0]
print(f"Remaining columns with missing values: {len(missing_nonzero)} out of {len(data.columns)} columns")
missing_nonzero.head()

Remaining columns with missing values: 2 out of 28 columns


,Missing Count,Missing %
Longitude,8,0.0
Latitude,8,0.0


### Training and Testing Sets
Now that most of the missing values have been handled, the training and testing splits can be generated. Per the task spec, the most revent month—in this case, May 2026—will be the testing set. In order to determine the optimal number of months for the training set, a quick linear regression will be run. Additionally, outlier capping for `LotSizeSquareFeet` and missing value handling for `Latitude` and `Longitude` will be implemented.

In [15]:
data.columns

Index(['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'CloseDate',
       'Latitude', 'Longitude', 'LivingArea', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'CloseMonth', 'CloseYear', 'logClosePrice'],
      dtype='object')

In [16]:
months= sorted(data['CloseMonth'].unique())

# Test set is always the most recent month (May 2026)
test_month = months[-1]
test_data = data[data['CloseMonth'] == test_month].copy()

def find_optimal_training_months(data, features, target="logClosePrice", max_months=5):
    """
    Determines the optimal number of historical months to use for training
    a Linear Regression model based on highest R² score.

    Parameters:
        data (pd.DataFrame): Dataset containing CloseMonth, features, and target
        features (list): List of feature column names
        target (str): Target variable
        max_months (int): Maximum number of historical months to test

    Returns:
        optimal_months (int): Number of months with highest R² score
        results (pd.DataFrame): R² scores for each training window
    """

    # Ensure chronological order
    data = data.sort_values("CloseMonth").copy()

    # Get unique months in chronological order
    months = sorted(data["CloseMonth"].unique())

    # Most recent month is always the test month
    test_month = months[-1]
    test = data[data["CloseMonth"] == test_month]

    results = []

    # Can't use more training months than exist before the test month
    max_months = min(max_months, len(months) - 1)

    for n_months in range(1, max_months + 1):

        # Previous n months before the test month
        train_months = months[-(n_months + 1):-1]

        train = data[data["CloseMonth"].isin(train_months)]

        X_train = train[features]
        y_train = train[target]

        X_test = test[features]
        y_test = test[target]

        model = LinearRegression()
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        r2 = r2_score(y_test, predictions)

        results.append({
            "Training Months": n_months,
            "Test Month": test_month,
            "R2 Score": r2
        })

    results = pd.DataFrame(results)

    optimal_months = results.loc[
        results["R2 Score"].idxmax(),
        "Training Months"
    ]

    return optimal_months, results

In [17]:
tuning_cols = data.select_dtypes(include="number").drop(columns=["logClosePrice", 'Longitude', 'Latitude']).columns.tolist()

# determing optimal window
n_months_train, results = find_optimal_training_months(data, features=tuning_cols)

print(results)
print(f'Optimal training window: {n_months_train} months')

   Training Months Test Month  R2 Score
0                1    2026-05  0.380801
1                2    2026-05  0.381679
2                3    2026-05  0.381387
3                4    2026-05  0.380313
4                5    2026-05  0.380236
Optimal training window: 2 months


In [18]:
# optimal number of months is 2 months, so set training set accordingly
train_months = months[-(n_months_train+1):-1]
train_data = data[data['CloseMonth'].isin(train_months)].copy()

print("Training months:", train_months)
print("Test month:", test_month)
print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")

Training months: [Period('2026-03', 'M'), Period('2026-04', 'M')]
Test month: 2026-05
Training rows: 21,379
Test rows: 11,040


In [19]:
# fillna in Latitude and Longitude with county specific medians
for coord in ['Latitude', 'Longitude']:
    # calculate on train set, apply on test set
    county_medians = train_data.groupby('CountyOrParish')[coord].median()
    global_median = train_data[coord].median()

    # fillna with county specific medians
    train_data[coord] = train_data[coord].fillna(train_data['CountyOrParish'].map(county_medians))
    test_data[coord] = test_data[coord].fillna(test_data['CountyOrParish'].map(county_medians))

    # fillna with global medians if county does not exist
    train_data[coord] = train_data[coord].fillna(global_median)
    test_data[coord] = test_data[coord].fillna(global_median)

In [20]:
# visualize missing values for TRAIN_DATA after dropping columns
missing = train_data.isnull().sum()
missing_pct = (missing / len(train_data) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Show columns that have at least some missing train_data
missing_nonzero = missing_df[missing_df['Missing Count'] > 0]
print(f"Columns with missing values: {len(missing_nonzero)} out of {len(train_data.columns)} columns")
missing_nonzero.head(30)

Columns with missing values: 0 out of 28 columns


,Missing Count,Missing %


In [21]:
# visualize missing values for TEST_DATA after dropping columns
missing = test_data.isnull().sum()
missing_pct = (missing / len(test_data) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Show columns that have at least some missing test_data
missing_nonzero = missing_df[missing_df['Missing Count'] > 0]
print(f"Columns with missing values: {len(missing_nonzero)} out of {len(test_data.columns)} columns")
missing_nonzero.head(30)

Columns with missing values: 0 out of 28 columns


,Missing Count,Missing %


In [22]:
# outlier capping for LotSizeSquareFeet (can be down now, since capping relied on percentile and data has been split)
low, high = 0, train_data['LotSizeSquareFeet'].quantile(0.995) # high cap calculated on train set, applied to test set to avoid leakage 
train_data = train_data[train_data['LotSizeSquareFeet'].between(low, high)]
test_data = test_data[test_data['LotSizeSquareFeet'].between(low, high)]

In [23]:
# drop all date columns, since we don't need them anymore
train_data = train_data.drop(columns=['CloseMonth', 'CloseYear', 'CloseDate'])
test_data = test_data.drop(columns=['CloseMonth', 'CloseYear', 'CloseDate'])

### Target Encoding Categorical Features using Bayesian Smoothing
The data contains numerous categorical features, which need to be converted to numeric representations before the mdoel can use the data. Bayesian smoothing will be used, replacing each value with the mean value of `logClosePrice`. The smoothing prevents data leakage and prevents overfitting on rare categories.

In [24]:
# target encode remaining categorical columns 
def target_encode(train_df, test_df, col, target='logClosePrice', smoothing=50):
    """
    Target-encode a categorical column using training data only.
    Uses global mean smoothing to handle rare categories.
    """
    global_mean = train_df[target].mean()

    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['weight'] = stats['count'] / (stats['count'] + smoothing)
    stats['encoded'] = stats['weight'] * stats['mean'] + (1 - stats['weight']) * global_mean

    encoding_map = stats['encoded'].to_dict()

    train_encoded = train_df[col].map(encoding_map).fillna(global_mean)
    test_encoded = test_df[col].map(encoding_map).fillna(global_mean)

    return train_encoded, test_encoded, encoding_map

target_encode_cols = train_data.select_dtypes(include='O').columns.tolist()
maps = {}

# perform target encoding
for col in target_encode_cols:
    train_enc, test_enc, encoding_map = target_encode(train_data, test_data, col)
    train_data[col] = train_enc
    test_data[col] = test_enc
    maps[col] = encoding_map

train_data.info(verbose=False)
print()
test_data.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 21272 entries, 38663 to 101114
Columns: 25 entries, ViewYN to logClosePrice
dtypes: bool(7), float64(18)
memory usage: 3.2 MB

<class 'pandas.core.frame.DataFrame'>
Index: 10986 entries, 101147 to 124400
Columns: 25 entries, ViewYN to logClosePrice
dtypes: bool(7), float64(18)
memory usage: 1.7 MB


### Scale Numerical Features
To improve the linear model, a StandardScaler will be fitted on the training set and applied to the test set. Both scaled and unscaled versions will be saved; scaled version for the linear model, unscaled version for the tree-based models.

In [25]:
# initialize scaler
scaler = StandardScaler()

# define final feature columns
drop_from_features = ['ClosePrice', 'logClosePrice', 'CloseDate', 'CloseMonth', 'CloseYear', 'LotSizeSquareFeet', 'Latitude', 'Longitude']
feature_cols = [c for c in train_data.columns if c not in drop_from_features]

numeric_cols_to_scale = train_data[feature_cols].select_dtypes(include='number').columns.tolist()
print(f"Total feature columns: {len(feature_cols)}")
print(f"Numeric columns to scale: {len(numeric_cols_to_scale)}")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols, 1):
    dtype = train_data[col].dtype
    scale_tag = " [SCALED]" if col in numeric_cols_to_scale else ""
    print(f"  {i:2d}. {col} ({dtype}){scale_tag}")

Total feature columns: 21
Numeric columns to scale: 14

Feature list:
   1. ViewYN (bool)
   2. WaterfrontYN (bool)
   3. BasementYN (bool)
   4. PoolPrivateYN (bool)
   5. LivingArea (float64) [SCALED]
   6. CountyOrParish (float64) [SCALED]
   7. AttachedGarageYN (bool)
   8. ParkingTotal (float64) [SCALED]
   9. YearBuilt (float64) [SCALED]
  10. BathroomsTotalInteger (float64) [SCALED]
  11. City (float64) [SCALED]
  12. BedroomsTotal (float64) [SCALED]
  13. FireplaceYN (bool)
  14. Stories (float64) [SCALED]
  15. Levels (float64) [SCALED]
  16. MainLevelBedrooms (float64) [SCALED]
  17. NewConstructionYN (bool)
  18. GarageSpaces (float64) [SCALED]
  19. HighSchoolDistrict (float64) [SCALED]
  20. PostalCode (float64) [SCALED]
  21. AssociationFee (float64) [SCALED]


In [26]:
# save unscaled versions first (for tree-based models)
train_data.to_csv('data/model_sets/train_set_unscaled.csv', index=False)
test_data.to_csv('data/model_sets/test_set_unscaled.csv', index=False)

# fit scaler on training set
train_data[numeric_cols_to_scale] =scaler.fit_transform(train_data[numeric_cols_to_scale])
test_data[numeric_cols_to_scale] = scaler.transform(test_data[numeric_cols_to_scale])

print(f"Scaled feature means (should be ~0):")
print(train_data[numeric_cols_to_scale].mean().round(4).to_string())

Scaled feature means (should be ~0):
LivingArea               0.0
CountyOrParish           0.0
ParkingTotal            -0.0
YearBuilt                0.0
BathroomsTotalInteger   -0.0
City                     0.0
BedroomsTotal            0.0
Stories                 -0.0
Levels                  -0.0
MainLevelBedrooms        0.0
GarageSpaces             0.0
HighSchoolDistrict       0.0
PostalCode              -0.0
AssociationFee          -0.0


In [27]:
# save scaled versions (for linear models)
train_data.to_csv('data/model_sets/train_set_scaled.csv', index=False)
test_data.to_csv('data/model_sets/test_set_scaled.csv', index=False)

### Preprocessing Summary

| **Action** | **Details** |
|---|---|
| ***Data Loading*** | Loaded data, filtered for Single Family Residences |
| ***Dropping Columns*** | Droppped columns that were mostly empty, administrative columns, and columns that induced data leakage |
| ***Missing Values*** | Imputed with 0, median value, cross filled for `LotSizeSquareFeet`, county medians for `Latitude` and `Longtitude`, False for boolean columns, and "Unknown" for categorical columns |
| ***Outliers*** | Filtered out extreme values in various columns |
| ***Target Encoding*** | Used target encoding with bayesian smoothing to encode categorical columns (i.e. `City`, `PostalCode`, etc.), fitting only on training set | 
| ***Scaling*** | Used a StandardScaler on numeric features |
| ***Train/Test Split*** | Temporaly split the data into testing set (most revent month) and training set (optimal X months before, determined experimentally) |
